# 🚀 SparseChron — 4D Live Demo Runner

Proven live demo runner using the **347k Gaussian Dynamic Model**.

### Settings
- **Accelerator:** GPU T4
- **Internet:** ON

Run all 3 cells in order (~45s total).

In [ ]:
# 1. Setup & fast dependencies (~25s)
import os, subprocess, sys

REPO = "/kaggle/working/SparseChron"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/rajhodedara/SparseChron.git", REPO], check=True)

%cd /kaggle/working/SparseChron

# Checkout proven August codebase (exact sigmoid shaders for 347k model)
!git checkout ce7fb83

# Unpack precompiled CUDA kernels directly into PyTorch JIT cache (~2s)
!mkdir -p /root/.cache/torch_extensions
!curl -sL https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints/gsplat_cache.tar.gz | tar -xz -C /root/.cache/torch_extensions

# Fast install dependencies
WHEEL = "https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints/gsplat-1.5.3-py3-none-any.whl"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", WHEEL])
!pip install -q --no-deps viser tyro plyfile roma einops
!pip install -q timm lpips torchmetrics

print("\n🚀 Environment ready!")


In [ ]:
# 2. Ensure 347k Dynamic Model is downloaded (~15s)
import os, urllib.request

os.makedirs("/kaggle/working/outputs_mutant", exist_ok=True)
CKPT_PATH = "/kaggle/working/outputs_mutant/checkpoint_august_30000.ckpt"

if not os.path.exists(CKPT_PATH) or os.path.getsize(CKPT_PATH) < 50_000_000:
    print("Downloading 347k model (237 MB)...")
    urllib.request.urlretrieve(
        "https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints/checkpoint_august_30000.ckpt",
        CKPT_PATH
    )
    print("Downloaded!")
else:
    print(f"Model verified on disk: {os.path.getsize(CKPT_PATH)/1e6:.1f} MB")

print("Ready for viewer!")


In [ ]:
# 3. Launch 4D Viewer (Viser) + Public Tunnel
!pkill -9 -f interactive_viewer; !pkill -9 -f localtunnel; !pkill -9 -f cloudflared
import time, socket, subprocess, os

!curl -s ifconfig.me; echo ""

CKPT_PATH = "/kaggle/working/outputs_mutant/checkpoint_august_30000.ckpt"

# Launch viewer process in background
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
subprocess.Popen(
    ["python", "/kaggle/working/SparseChron/scripts/interactive_viewer.py", "--checkpoint", CKPT_PATH, "--port", "8080"],
    env=env
)

# Wait for Viser to open port 8080
print("Loading 347k Gaussians into GPU VRAM (waiting for port 8080)...")
for _ in range(30):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", 8080)) == 0:
            print("✅ 4D Viewer is LIVE on port 8080!\n")
            break
    time.sleep(1)

print("Launching Tunnel...")
# Launch Cloudflare Tunnel
if not os.path.exists("/kaggle/working/cloudflared"):
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /kaggle/working/cloudflared && chmod +x /kaggle/working/cloudflared
!/kaggle/working/cloudflared tunnel --url http://127.0.0.1:8080
